In [1]:
#SETUP
from datetime import datetime
from delta import DeltaTable
from pyspark.shell import spark

import ConnectionConfig as cc
cc.setupEnvironment()

In [2]:
spark = cc.startLocalCluster("Analyzing") # default 4
spark.getActiveSession()

In [3]:
#EXTRACT
dim_customer = spark.read.format("delta").load("spark-warehouse/dimCustomer")
dim_locks = spark.read.format("delta").load("spark-warehouse/dimLocks")           
fact_rides = spark.read.format("delta").load("spark-warehouse/factRidesS2")

In [4]:
#TRANSFORM
#TEMP VIEW MAKEN  
dim_customer.createOrReplaceTempView("dimCustomer")
dim_locks.createOrReplaceTempView("dimLocks")
fact_rides.createOrReplaceTempView("factRides")

## Wat is de invloed van de woonplaats van de gebruikers op het gebruik van de vehicles?

We kunnen deze vraag beantwoorden door te kijken naar het aantal ritten en de gemiddelde ritafstand, gegroepeerd per woonplaats (stad) van de gebruikers. Dit geeft inzicht in welke steden meer gebruik maken van de fietsen en of de afstanden verschillen per locatie.



In [17]:
influence_residence = spark.sql("""
SELECT 
    dc.city AS woonplaats,
    COUNT(fr.rideId) AS totaal_aantal_ritten,
    AVG(fr.rideDistance) AS gemiddelde_ritafstand,
    COUNT(DISTINCT fr.customerSK) AS unieke_gebruikers,
    COUNT(fr.rideId) / COUNT(DISTINCT fr.customerSK) AS ritten_per_gebruiker
FROM 
    factRides AS fr
JOIN 
    dimCustomer AS dc ON fr.customerSK = dc.customer_SK
GROUP BY 
    dc.city
ORDER BY 
    totaal_aantal_ritten DESC;

     """)

influence_residence.show()


+--------------------+--------------------+---------------------+-----------------+--------------------+
|          woonplaats|totaal_aantal_ritten|gemiddelde_ritafstand|unieke_gebruikers|ritten_per_gebruiker|
+--------------------+--------------------+---------------------+-----------------+--------------------+
|           Antwerpen|              554496| 2.582786317984922...|             8185|   67.74538790470373|
|             Schoten|              137392| 2.773419127795431...|             1985|   69.21511335012595|
|Beveren-Waas/Haas...|              135227| 2.774555526936544...|             2032|   66.54872047244095|
| Wilrijk (Antwerpen)|              121475| 3.179163188129609...|             1836|    66.1628540305011|
|  Deurne (Antwerpen)|              113640| 2.969497925356933...|             1659|     68.499095840868|
|          Brasschaat|              100074| 2.771924181545704...|             1474|   67.89280868385346|
|    Kapellen (Antw.)|               98391| 2.774589307

## We willen voorspellen welke sloten preventief onderhoud nodig hebben. Bekijk hoe vaak slotnummers relatief gezien gebruikt worden.


Om sloten te identificeren die mogelijk preventief onderhoud nodig hebben, kijken we naar hoe vaak elk slot wordt gebruikt, zowel als start- als eindslot.

In [9]:
locks_maintenance =  spark.sql(""" 
SELECT 
    dl.stationnr AS stationnummer,
    dl.stationlocknr AS slotnummer,
    COUNT(fr.rideId) AS gebruiksfrequentie,
    RANK() OVER (ORDER BY COUNT(fr.rideId) DESC) AS gebruiks_ranking
FROM 
    factRides AS fr
JOIN 
    dimLocks AS dl ON fr.startLockSK = dl.lockSK OR fr.endLockSK = dl.lockSK
GROUP BY 
    dl.stationnr, dl.stationlocknr
ORDER BY 
    gebruiksfrequentie DESC;
    """)
locks_maintenance.show()


+-------------+----------+------------------+----------------+
|stationnummer|slotnummer|gebruiksfrequentie|gebruiks_ranking|
+-------------+----------+------------------+----------------+
|          103|        18|             17825|               1|
|          145|        18|             17754|               2|
|          115|        18|             17675|               3|
|          155|        36|             17663|               4|
|          228|        36|             17629|               5|
|          052|        18|             17622|               6|
|          062|        18|             17622|               6|
|          080|        18|             17605|               8|
|          102|        18|             17568|               9|
|          076|        18|             17543|              10|
|          064|        18|             17520|              11|
|          053|        18|             17509|              12|
|          137|        36|             17481|          

## Als een klant zijn abonnement stopzet, willen we kunnen voorspellen op welke stations dit het meeste effect zal hebben.

Hier analyseren we het gebruik van stations door klanten die hun abonnement hebben stopgezet, zodat we kunnen zien op welke stations het effect van opzeggingen het grootst is.

In [10]:
subscription_effect= spark.sql("""
SELECT 
    dl.stationnr AS stationnummer,
    COUNT(fr.rideId) AS aantal_ritten_door_stopgezette_klanten
FROM 
    factRides AS fr
JOIN 
    dimLocks AS dl ON fr.startLockSK = dl.lockSK OR fr.endLockSK = dl.lockSK
JOIN 
    dimCustomer AS dc ON fr.customerSK = dc.customer_SK
WHERE 
    dc.scd_end IS NOT NULL -- Alleen klanten die hun abonnement hebben stopgezet
GROUP BY 
    dl.stationnr
ORDER BY 
    aantal_ritten_door_stopgezette_klanten DESC; 
    """)
subscription_effect.show()

+-------------+--------------------------------------+
|stationnummer|aantal_ritten_door_stopgezette_klanten|
+-------------+--------------------------------------+
|          227|                                 59838|
|          225|                                 59667|
|          305|                                 59082|
|          307|                                 57413|
|          033|                                 57059|
|          123|                                 55668|
|          043|                                 55470|
|          059|                                 55453|
|          137|                                 55172|
|          061|                                 55091|
|          203|                                 54966|
|          127|                                 54848|
|          055|                                 54749|
|          050|                                 54576|
|          038|                                 54545|
|         

## Welke klanten maken de meeste en langste ritten?

We willen weten welke klanten het vaakst en het langst voertuigen gebruiken.



In [11]:
langste_rides = spark.sql("""
SELECT 
    dc.customer_SK AS klant_id,
    COUNT(fr.rideId) AS totaal_aantal_ritten,
    SUM(fr.rideDistance) AS totale_afgelegde_afstand,
    AVG(fr.rideDistance) AS gemiddelde_afstand_per_rit
FROM 
    factRides AS fr
JOIN 
    dimCustomer AS dc ON fr.customerSK = dc.customer_SK
GROUP BY 
    dc.customer_SK
ORDER BY 
    totaal_aantal_ritten DESC, gemiddelde_afstand_per_rit DESC
LIMIT 10;
""")
langste_rides.show()

+----------+--------------------+------------------------+--------------------------+
|  klant_id|totaal_aantal_ritten|totale_afgelegde_afstand|gemiddelde_afstand_per_rit|
+----------+--------------------+------------------------+--------------------------+
|8589984478|                 140|    402.4890349154878...|      2.874921677967770...|
|8589949969|                 140|    367.7664445317839...|      2.626903175227028...|
|8589993683|                 138|    408.6817360002742...|      2.961461855074450...|
|8589974702|                 137|    389.7713693520779...|      2.845046491621007...|
|8589978694|                 136|    412.7756704636628...|      3.035115223997521...|
|8589950406|                 135|    378.3323829752836...|      2.802462096113212...|
|8589964433|                 134|    379.1300939757854...|      2.829329059520786...|
|8589934907|                 134|    375.2959548533484...|      2.800716080995137...|
|8589969867|                 134|    346.8421309519559

### Welke abonnementsvormen leiden tot het hoogste aantal ritten en de langste ritafstanden per klant?


In [15]:
analyse_stations = spark.sql("""
SELECT 
    dc.subscription_type AS abonnementsvorm,
    COUNT(fr.rideId) AS totaal_aantal_ritten,
    AVG(fr.rideDistance) AS gemiddelde_ritafstand,
    COUNT(DISTINCT fr.customerSK) AS unieke_gebruikers,
    COUNT(fr.rideId) / COUNT(DISTINCT fr.customerSK) AS ritten_per_gebruiker
FROM 
    factRides AS fr
JOIN 
    dimCustomer AS dc ON fr.customerSK = dc.customer_SK
GROUP BY 
    dc.subscription_type
ORDER BY 
    totaal_aantal_ritten DESC;

    """)
analyse_stations.show()


+---------------+--------------------+---------------------+-----------------+--------------------+
|abonnementsvorm|totaal_aantal_ritten|gemiddelde_ritafstand|unieke_gebruikers|ritten_per_gebruiker|
+---------------+--------------------+---------------------+-----------------+--------------------+
|           JAAR|             2755947| 2.775013256010756...|            34620|   79.60563258232236|
|          MAAND|               10871| 2.781569308340108...|             4858|  2.2377521613832854|
|            DAG|                1560| 2.832285063515525...|             1304|   1.196319018404908|
+---------------+--------------------+---------------------+-----------------+--------------------+

